# 01 · Download and save the project datasets

**Project:** Measuring Sentiment Loss in Chinese Internet Slang in Translation

This notebook downloads the three datasets from our proposal and saves them in two places:

| Folder | What goes there |
|---|---|
| `data/raw/` | The files exactly as downloaded (we never edit these by hand) |
| `data/processed/` | Clean, tidy CSV/JSON versions that later notebooks will read |


| Dataset | Role in the project | Source |
|---|---|---|
| **SMP2020-EWECT** | Main corpus: Weibo posts with 6 emotion labels | GitHub |
| **HADQAET** | Human-annotated MT benchmark | `surrey-nlp/HADQAET` |
| **CHEER** | ~1,100 Chinese internet buzzwords with definitions, used to detect slang | `SCUNLP/Buzzword` |

## Sections ##
- 0: Setup (imports, folders)
- 1: SMP2020-EWECT (download, quick EDA, save)
- 2: HADQAET (download, quick EDA, save)
- 3: CHEER (download, quick EDA, save)
- 4: Simple EDA (how many SMP2020 posts contain a CHEER word)
- 5: Summary of Downloads (folder, what saved, what size)

## 0: Setup

In [14]:
%pip install openpyxl #will need this to read the Excel notebook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import ast
import json
import os

import pandas as pd
import requests

# Show more text per cell when we print tables (Chinese posts can be long)
pd.set_option("display.max_colwidth", 80)

The following code makes the folders for the raw data, make sure the paths that print are correct!!

In [2]:
# This notebook is in  'repo>/notebooks/'
# so the top folder of the repo ("project root") is one level up.
current_folder = os.getcwd()

if os.path.basename(current_folder) == "notebooks": #if currently in notebooks, go to project root src: https://stackoverflow.com/questions/22272003/what-is-the-difference-between-os-path-basename-and-os-path-dirname 
    project_root = os.path.dirname(current_folder)
else: #otherwise already in root
    project_root = current_folder


raw_folder = os.path.join(project_root, "data", "raw") # Folder for untouched downloads
processed_folder = os.path.join(project_root, "data", "processed") #folder for cleaned outputs

#A sub-folder per dataset
smp_raw_folder = os.path.join(raw_folder, "smp2020_ewect")
hadqaet_raw_folder = os.path.join(raw_folder, "hadqaet")
cheer_raw_folder = os.path.join(raw_folder, "cheer")

# Create each folder.
folders_to_create = [smp_raw_folder, hadqaet_raw_folder, cheer_raw_folder, processed_folder]
for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True) # exist_ok=True means "don't complain if the folder is already there". src: https://stackoverflow.com/questions/600268/mkdir-p-functionality-in-python

print("Project root: ", project_root)
print("Raw data: ", raw_folder)
print("Processed data: ", processed_folder)

Project root:  /Users/cierrajagan/github/W266-slang-emotion-mt
Raw data:  /Users/cierrajagan/github/W266-slang-emotion-mt/data/raw
Processed data:  /Users/cierrajagan/github/W266-slang-emotion-mt/data/processed


In [3]:
def download_file(url, save_path):
    ''' Helper function: download one file
    url -> the web address for the file
    save_path -> where to save it on our computer 
    '''
    file_name = os.path.basename(save_path)

    # Download the file
    print("  downloading: ", file_name) 
    response = requests.get(url, timeout=60) #src: https://realpython.com/python-requests/

    # Check that the download worked (status code 200 = worked)
    print('Status code: ', response.status_code)

    # Save the file. "wb" means "write bytes" (so it saves it exactly as received)
    with open(save_path, "wb") as file:
        file.write(response.content)

## 1: SMP2020-EWECT 

Weibo posts, each labelled with one of six emotions: happy, angry, sad, fear, surprise, neutral.

Comes in two subsets:
- **usual**: random everyday posts (27,768 train / 2,000 eval / 5,000 test = **34,768**; this is the "~35k" in our proposal)
- **virus**: COVID-19 posts (8,606 / 2,000 / 3,000)

Got files from a public GitHub copy (BrownSweater/BERT_SMP2020-EWECT). Each file is a JSON list.

We download both subsets and mark each row with its subset, so we can decide later whether to use virus at all.

In [4]:
# idea from https://stackoverflow.com/questions/23464138/downloading-and-accessing-data-from-github-python

# The commit we download from 
smp_commit = "e928cec001824668999ed02326aacbea0d369563"
# Every file lives in the same online place
smp_base_url = "https://raw.githubusercontent.com/BrownSweater/BERT_SMP2020-EWECT/" + smp_commit + "/data/raw/"

# The six files: 2 subsets (usual, virus) 3 splits per subset (train, eval, test)
smp_file_names = [
    "usual_train.txt",
    "usual_eval_labeled.txt",
    "usual_test_labeled.txt",
    "virus_train.txt",
    "virus_eval_labeled.txt",
    "virus_test_labeled.txt",
]

print("Downloading SMP2020-EWECT:")
for file_name in smp_file_names: 
    url = smp_base_url + file_name
    save_path = os.path.join(smp_raw_folder, file_name) #save each file using the helper function
    download_file(url, save_path)

  downloading:  usual_train.txt
Status code:  200
  downloading:  usual_eval_labeled.txt
Status code:  200
  downloading:  usual_test_labeled.txt
Status code:  200
  downloading:  virus_train.txt
Status code:  200
  downloading:  virus_eval_labeled.txt
Status code:  200
  downloading:  virus_test_labeled.txt
Status code:  200


In [5]:
# Load all files into one table

# Each row is a dictionary like:
#   {"post_id": ..., "subset": ..., "split": ..., "text": ..., "label": ...}
smp_rows = []

for file_name in smp_file_names:
    # Get the subset (unique/virus) and the split (train/eval/test). Ex: "usual_eval_labeled.txt" -> "usual_eval_labeled" -> ["usual", "eval", "labeled"]
    name_without_ending = file_name.replace(".txt", "")
    name_parts = name_without_ending.split("_")
    subset = name_parts[0]   # "usual"/"virus"
    split = name_parts[1]    # "train"/"eval"/"test"

    # Open and read the JSON file using UTF-8
    file_path = os.path.join(smp_raw_folder, file_name)
    with open(file_path, encoding="utf-8-sig") as file: 
        posts = json.load(file) # posts is a list of dictionaries

    # Add each post to our list of rows
    for post in posts:
        original_id = post["id"]

        # The original ids restart at 1 in every file, so they are NOT unique, so we need to make our own unique id, e.g. "usual_train_00001".
        # zfill(5) pads the number with zeros to 5 digits to standardize: 1 -> "00001" 
        post_id = subset + "_" + split + "_" + str(original_id).zfill(5)

        row = {
            "post_id": post_id,
            "subset": subset,
            "split": split,
            "orig_id": original_id,
            "text": post["content"],
            "label": post["label"],
        }
        smp_rows.append(row)
    print(file_name, "->", len(posts), "posts")

# Turn the list of rows into a pandas table so it's easier to work with
smp = pd.DataFrame(smp_rows)

print()
print("Total posts:", len(smp))
smp.head()

usual_train.txt -> 27768 posts
usual_eval_labeled.txt -> 2000 posts
usual_test_labeled.txt -> 5000 posts
virus_train.txt -> 8606 posts
virus_eval_labeled.txt -> 2000 posts
virus_test_labeled.txt -> 3000 posts

Total posts: 48374


,post_id,subset,split,orig_id,text,label
0,usual_train_00001,usual,train,1,气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车,angry
1,usual_train_00002,usual,train,2,妞妞啊，今天又承办了一个发文登记文号是126~嘻~么么哒~晚安哟,happy
2,usual_train_00003,usual,train,3,这里还值得注意另一个事实，就是张鞠存原有一个东溪草堂为其读书处。,neutral
3,usual_train_00004,usual,train,4,这在前华约国家(尤其是东德)使用R-73的首次联合演习期间，被一些北约组织的飞行员所证实。,neutral
4,usual_train_00005,usual,train,5,TinyThief上wii了？！,surprise


In [6]:
# What share of each subset has each emotion?
# If some emotions are rare, plain accuracy can be misleading later, which is why we plan to report macro-F1 too.

for subset in ["usual", "virus"]: #get how many posts for usual/virus
    subset_posts = smp[smp["subset"] == subset]
    total = len(subset_posts)
    print(f"Subset: {subset} ({str(total)} posts)")

    label_counts = subset_posts["label"].value_counts() # count of each emotion label

    for label in label_counts.index:
        count = label_counts[label]
        percent = count / total * 100
        print(f"   {label}: {count} posts = {round(percent, 1)}%")
    print()

Subset: usual (34768 posts)
   angry: 10438 posts = 30.0%
   neutral: 7159 posts = 20.6%
   happy: 6788 posts = 19.5%
   sad: 6236 posts = 17.9%
   surprise: 2630 posts = 7.6%
   fear: 1517 posts = 4.4%

Subset: virus (13606 posts)
   happy: 6886 posts = 50.6%
   neutral: 2456 posts = 18.1%
   angry: 2099 posts = 15.4%
   sad: 1033 posts = 7.6%
   fear: 820 posts = 6.0%
   surprise: 312 posts = 2.3%



In [7]:
# Check for empty and duplicate posts

empty_count = 0
seen_texts = set() # every text we've seen so far
duplicate_count = 0

for text in smp["text"]:
    # Count posts that are blank (only spaces or nothing)
    if text.strip() == "":
        empty_count = empty_count + 1

    # Count posts whose exact text we've already seen
    if text in seen_texts: #if test already exists in seen_text set, add to the duplicate_count
        duplicate_count = duplicate_count + 1
    else: #otherwise add the new text to the seen_text set
        seen_texts.add(text)

print("Empty posts:", empty_count)
print("Repeated posts (same text seen before):", duplicate_count)

Empty posts: 2
Repeated posts (same text seen before): 2680


In [8]:
# Save the combined table. ()"utf-8-sig" makes Chinese characters display correctly if you open the CSV in Excel)

smp_output_path = os.path.join(processed_folder, "smp2020_ewect.csv")
smp.to_csv(smp_output_path, index=False, encoding="utf-8-sig")
print("Saved", smp_output_path)

Saved /Users/cierrajagan/github/W266-slang-emotion-mt/data/processed/smp2020_ewect.csv


## 2: HADQAET (human-annotated MT benchmark)

Built on top of SMP2020-EWECT: 5,538 Weibo posts translated with Google Translate with human annotations of emotion-preservation errors.

Two Excel files:
- annotated_dataset.xlsx
  - **Sheet1**: source, MT, emotion_labels, error_types, error_severity (5,538 rows)
  - **Sheet2**: emotion keywords in the source and the translation
- post-edits70p.xlsx: the ~70% subset released with human post-edited reference translations (4,038 rows, ~3,000 with post-edits)

In [9]:
# Same idea as the SMP2020 downlaod

# The commit we download from 
hadqaet_commit = "4a515d686666cf7bdb8c008a8a9c26e77fda8280"
# Every file lives in the same online place
hadqaet_base_url = "https://raw.githubusercontent.com/surrey-nlp/HADQAET/" + hadqaet_commit + "/data/"

hadqaet_file_names = ["annotated_dataset.xlsx", "post-edits70p.xlsx"]

print("Downloading HADQAET:")
for file_name in hadqaet_file_names:
    url = hadqaet_base_url + file_name
    save_path = os.path.join(hadqaet_raw_folder, file_name)
    download_file(url, save_path)

  downloading:  annotated_dataset.xlsx
Status code:  200
  downloading:  post-edits70p.xlsx
Status code:  200


In [10]:
# Load the Excel sheets
annotated_path = os.path.join(hadqaet_raw_folder, "annotated_dataset.xlsx")
post_edits_path = os.path.join(hadqaet_raw_folder, "post-edits70p.xlsx")

# sheet_name tells pandas which tab of the Excel file to read
hadqaet = pd.read_excel(annotated_path, sheet_name="Sheet1")# main annotation table
hadqaet_keywords = pd.read_excel(annotated_path, sheet_name="Sheet2") # emotion keyword pairs
hadqaet_post_edits = pd.read_excel(post_edits_path)# only has one tab

print(f"Main table: {len(hadqaet)} rows")
print(f"   columns: {list(hadqaet.columns)}")
print(f"Keyword table: {len(hadqaet_keywords)} rows")
print(f"   columns: {list(hadqaet_keywords.columns)}")
print(f"Post-edit table: {len(hadqaet_post_edits)} rows")
print(f"   columns: {list(hadqaet_post_edits.columns)}")

hadqaet.head()

Main table: 5538 rows
   columns: ['source', 'MT', 'emotion_labels', 'error_types', 'error_severity']
Keyword table: 2915 rows
   columns: ['keywords-source', 'keywords-target']
Post-edit table: 4038 rows
   columns: ['source', 'MT', 'emotion_labels', 'error_types', 'error_severity', 'post-edit']


,source,MT,emotion_labels,error_types,error_severity
0,早上的我，竟然也变成了一个无理取闹的人……,"In the morning, I turned into an unreasonable person...",sadness,Omission,Major
1,听故事的时候，你总喜欢眼巴巴的问，后来呢，可是当后来，自己成为讲故事的人，才发现，后来故事和话语就在嘴边，后来眼泪几度泪凝于睫，是真的说不下去了。。,"When listening to a story, you always like to ask questions, but later, when...",sadness,Mistranslation; Mistranslation,Critical; Major
2,睡得好好的，要去别的学校听课，天啊！,"Sleep well, go to another school to attend classes, oh my god!",anger,Mistranslation,Critical
3,管理学真是水的一比，努力的想听，依然坚持不过一分钟……考研怎么办呀,"Management is really a comparison of water. I want to listen hard, but I sti...",anger,Mistranslation,Critical
4,想去日本台湾冰岛英国北欧，签证说no，小短假也说no,"I want to go to Japan, Taiwan, Iceland, the United Kingdom, Northern Europe,...",sadness,NaN,NaN


In [12]:
# Check the counts for each emotion
print("Emotion labels:")
print(hadqaet["emotion_labels"].value_counts())
print()

# error_severity is empty when annotators found no error
print("Most common error severity values:")
print(hadqaet["error_severity"].value_counts(dropna=False).head(10)) # need dropna=False to count the empty ones (where there is no error)
print()

# Check how many rows in the post-edit file actually have a post-edit
post_edit_count = 0
for post_edit in hadqaet_post_edits["post-edit"]:
    if pd.isna(post_edit): # True for empty cells, so skip
        continue
    if str(post_edit).strip() == "": #skip cells that only contain spaces
        continue
    post_edit_count = post_edit_count + 1

print(f"Rows with a post-edited translation: {post_edit_count} of {len(hadqaet_post_edits)}")

Emotion labels:
emotion_labels
anger       2064
joy         1403
sadness     1237
surprise     540
fear         294
Name: count, dtype: int64

Most common error severity values:
error_severity
NaN                             2760
Critical                         906
Major                            612
Minor                            450
Critical; Critical               204
Major; Critical                  116
Major; Major                      88
Critical; Major                   83
Minor; Major                      55
Critical; Critical; Critical      32
Name: count, dtype: int64

Rows with a post-edited translation: 2952 of 4038


In [13]:
# IMPORTANT CHECK: do HADQAET posts also appear in SMP2020-EWECT?

# HADQAET's Chinese posts were picked FROM SMP2020-EWECT. If the same post
# shows up in both the data we tune on and the data we test on, we could have data leakage (which would inflate our results)

# For each HADQAET post, find the matching SMP post

# Build a lookup dictionary: SMP text -> SMP post_id
text_to_post_id = {}
for index, row in smp.iterrows():
    text = row["text"].strip()
    # If the same text appears twice in SMP, keep the first one (had some duplicates when I checked before)
    if text not in text_to_post_id:
        text_to_post_id[text] = row["post_id"]

# Look up each HADQAET source sentence
matched_post_ids = []
for source_text in hadqaet["source"]:
    source_text = str(source_text).strip() #use .strip() so we don't get messed up by whitespace or anything
    if source_text in text_to_post_id:
        matched_post_ids.append(text_to_post_id[source_text])
    else:
        matched_post_ids.append(None)   # no match found

#save the matches as a new column
hadqaet["smp_post_id"] = matched_post_ids

#Count matches, and which SMP subset/split they came from
match_count = 0
where_they_came_from = {}
for post_id in matched_post_ids:
    if post_id is None:
        continue
    match_count = match_count + 1

    # post_id looks like "subset_split_00001" -> so take the first two parts
    parts = post_id.split("_")
    subset_and_split = parts[0] + " " + parts[1]
    if subset_and_split not in where_they_came_from:
        where_they_came_from[subset_and_split] = 0
    where_they_came_from[subset_and_split] = where_they_came_from[subset_and_split] + 1

print(f"HADQAET posts found word-for-word in SMP2020-EWECT: {match_count} of {len(hadqaet)}")
print()
print("Where the matches come from in SMP:")
for subset_and_split in where_they_came_from:
    print(f"   {subset_and_split}: {where_they_came_from[subset_and_split]}")

HADQAET posts found word-for-word in SMP2020-EWECT: 5537 of 5538

Where the matches come from in SMP:
   usual test: 3750
   usual train: 341
   usual eval: 1446


In [14]:
# Save the three HADQAET tables as CSV files
hadqaet.to_csv(os.path.join(processed_folder, "hadqaet_annotated.csv"), index=False, encoding="utf-8-sig")
hadqaet_keywords.to_csv(os.path.join(processed_folder, "hadqaet_keywords.csv"), index=False, encoding="utf-8-sig")
hadqaet_post_edits.to_csv(os.path.join(processed_folder, "hadqaet_post_edits.csv"), index=False, encoding="utf-8-sig")
print("Saved hadqaet_annotated.csv, hadqaet_keywords.csv, hadqaet_post_edits.csv")

Saved hadqaet_annotated.csv, hadqaet_keywords.csv, hadqaet_post_edits.csv


## 3: CHEER (Chinese internet buzzword dictionary)

Each buzzword has a short definition, a longer ground_truth description, about 30 example posts (examples), and definitions that different LLMs generated (which we don't need right now).

The authors saved the dataset as a Python file (data_cheating_filtered.py) that contains one giant dictionary.


We don't import that file, because importing a file runs any code inside it (ask me how I know...because I did it a million times trying to download it lol!). Instead use ast.literal_eval, which only reads plain data (text, lists, dictionaries) and doesn't run code! Src: https://medium.com/@aniruddhapal/the-power-of-the-ast-literal-eval-method-in-python-8fb4014a2574

In [15]:
cheer_commit = "1ab8c41777b3cca6fb1f179751a420c948d944a3"
cheer_url = "https://raw.githubusercontent.com/SCUNLP/Buzzword/" + cheer_commit + "/data/data_cheating_filtered.py"
cheer_raw_path = os.path.join(cheer_raw_folder, "data_cheating_filtered.py")

print("Downloading CHEER:")
download_file(cheer_url, cheer_raw_path)

  downloading:  data_cheating_filtered.py
Status code:  200


In [16]:
# Read the whole file as one long piece of text
with open(cheer_raw_path, encoding="utf-8") as file:
    file_text = file.read()

# Reads the text as Python code WITHOUT running it, and gives us a "tree" describing what the code says.
tree = ast.parse(file_text)

#The file has one statement -> dataclear = {...}
first_statement = tree.body[0]
variable_name = first_statement.targets[0].id
print("Variable name in file:", variable_name)

#literal_eval turns the {...} on the right side into a dictionary
cheer_dict = ast.literal_eval(first_statement.value)
print("Number of buzzwords:", len(cheer_dict))

Variable name in file: dataclear
Number of buzzwords: 1127


In [18]:
# Look at one entry so we know what it looks like
# I orginally coded with a loop so I could look at a few, but only showing one now to save space in notebook
for buzzword in cheer_dict:
    info = cheer_dict[buzzword]
    print("Example buzzword:", buzzword)
    print("Fields:", list(info.keys()))
    print("Definition:", info["definition"])
    print("Number of example posts:", len(info["examples"]))
    break # stop after the first one

Example buzzword: 我有个朋友
Fields: ['examples', 'definition', 'ground_truth', 'Gemini_15_pro_KEY', 'Gemini_15_flash_KEY', 'Qwen2_7b_KEY', 'GPT_4o_KEY', 'GPT_4o_mini_KEY', 'Qwen2_72b_KEY', 'Gemini_15_pro_KEY_score']
Definition: 指说话者为了避免尴尬或显得突兀，假借“朋友”的名义，委婉地表达自己的观点、疑问或经历的一种说法。
Number of example posts: 26


After turning dictionary into table, here are what the columns mean: 
- buzzword -> word
- definition -> short definition in Chinese
- description -> longer definition in Chinese
- n_examples -> how many examples it has
- n_chars -> how many characters the buzzword has

In [ ]:
# Turn the dictionary into a table

cheer_rows = []

for buzzword in cheer_dict:
    info = cheer_dict[buzzword]

    row = {
        "buzzword": buzzword,
        "definition": info["definition"],
        "description": info["ground_truth"],
        "n_examples": len(info["examples"]), 
        "n_chars": len(buzzword)
    }
    cheer_rows.append(row)

cheer = pd.DataFrame(cheer_rows) #pandas df to make it easier to work with

# Very short buzzwords (like 1 character) could match A LOT of posts by accident when we look for slang later, so it's worth checking the length of them
shortest = cheer["n_chars"].min()
longest = cheer["n_chars"].max()

print(f"Shortest buzzword: {shortest} characters")
print(f"Longest buzzword: {longest} characters")

cheer.head()

Shortest buzzword: 2 characters
Longest buzzword: 17 characters


,buzzword,definition,description,n_examples,n_chars
0,我有个朋友,指说话者为了避免尴尬或显得突兀，假借“朋友”的名义，委婉地表达自己的观点、疑问或经历的一种说法。,通常指某些人想问一个问题，或者想举个很贴切场景的例子，但是碍于面子，不好意思直接开口，于是就虚构一个朋友，假借朋友的名义提问,26,5
1,白人饭,指一种源自欧美，以生冷食材为主，无需过多加工，高纤维、高蛋白、低热量，且味道较为寡淡的简餐。常带有一定的戏谑意味。,起初是一些IP定位在欧洲国家的网友分享了自己同事或自己的午餐。网络上所谓的白人饭，指的就是一种欧美风格的餐食。后来又以其高纤维素、高蛋白质、低热量的特点...,54,3
2,半场开香槟,指在比赛中取得暂时领先就得意忘形，最终导致失败的行为。,在竞技比赛中，一方因为优势过大而得意忘形，最终被人翻盘的情况,39,5
3,抱一丝,网络用语，谐音“不好意思”，表示歉意或谦虚。,“不好意思啊”的谐音,58,3
4,背刺我,原指游戏中角色从背后攻击敌人，造成额外伤害的技能或行为，现比喻在别人毫无防备的情况下，暗中进行的背叛、陷害或打击。,在游戏中，指某个职业、英雄的技能，简单的说就是“背后一击”，在别人不注意的时候给予一个致命的打击。 如暴雪魔兽地图DOTA中英雄—隐形刺客力丸的技能，英...,26,3


In [21]:
# Save CHEER
# ---------------------------------------------------------------
# The word list as a CSV (small and easy to look at)
cheer.to_csv(os.path.join(processed_folder, "cheer_lexicon.csv"), index=False, encoding="utf-8-sig")

# 2) The example posts as JSON:  buzzword -> list of example posts
cheer_examples = {}
for buzzword in cheer_dict:
    cheer_examples[buzzword] = cheer_dict[buzzword]["examples"]

examples_path = os.path.join(processed_folder, "cheer_examples.json")
with open(examples_path, "w", encoding="utf-8") as file:
    json.dump(cheer_examples, file, ensure_ascii=False, indent=1)

print("Saved cheer_lexicon.csv and cheer_examples.json")

Saved cheer_lexicon.csv and cheer_examples.json


## 4: First look: how many SMP posts contain a CHEER buzzword?

This is a simple check: does the buzzword appear anywhere inside the post's text? 

It only tells us roughly how big the "slang" group will be. Proper slang detection (words that are also normal vocabulary, etc.) belongs in the next notebook.

In [22]:
# Get the list of buzzwords
buzzwords = list(cheer["buzzword"])

# Only look at the 'usual' subset (our main corpus)
usual = smp[smp["subset"] == "usual"].copy()

# For each post, check every buzzword. This takes about a minute.
found_lists = [] # for each post, the list of buzzwords found in it
has_slang = [] # for each post, True if at least one buzzword was found

for text in usual["text"]:
    found_in_this_post = []
    for word in buzzwords:
        if word in text: # if the buzzword is somewhere inside the text, append word
            found_in_this_post.append(word)
    found_lists.append(found_in_this_post) #list of buzzords found in this post
    has_slang.append(len(found_in_this_post) > 0) #True if the number of buzzwords found is bigger than 0 (indicating there is slang)

usual["buzzwords_found"] = found_lists 
usual["has_slang"] = has_slang

slang_count = sum(has_slang) #since true = 1 and false = 0, can use the sum for the number of trues 
percent = slang_count / len(usual) * 100
print("'usual' posts with at least one buzzword:", slang_count, "of", len(usual), "(" + str(round(percent, 1)) + "%)")

'usual' posts with at least one buzzword: 885 of 34768 (2.5%)


In [24]:
# Which buzzwords match most often?

buzzword_counts = {}
for found_in_this_post in found_lists: #each post's list
    for word in found_in_this_post: #each word in that list
        if word not in buzzword_counts: # if we haven't seen the word before, add it to the list
            buzzword_counts[word] = 0
        buzzword_counts[word] = buzzword_counts[word] + 1 #add a count if we see it

# Sort from most to least common
sorted_buzzwords = sorted(buzzword_counts, key=buzzword_counts.get, reverse=True)

print("Top 20 matched buzzwords:")
for word in sorted_buzzwords[:20]:
    print("   ", word, ":", buzzword_counts[word])

Top 20 matched buzzwords:
    我也是醉了 : 93
    狗带 : 52
    么么哒 : 49
    给力 : 30
    坑爹 : 30
    熊孩子 : 27
    剁手 : 26
    中毒 : 25
    存在感 : 22
    作死 : 22
    玻璃心 : 21
    装逼 : 20
    泪流满面 : 16
    脑残粉 : 14
    根本停不下来 : 13
    撕逼 : 12
    感人 : 11
    虐狗 : 11
    安利 : 11
    有生之年 : 11


In [ ]:
# Do slang posts have different emotions from non-slang posts?

slang_posts = usual[usual["has_slang"] == True]
no_slang_posts = usual[usual["has_slang"] == False]

labels = ["angry", "happy", "sad", "fear", "surprise", "neutral"]

print("label      slang %    no-slang %")

for label in labels:
    slang_percent = (slang_posts["label"] == label).sum() / len(slang_posts) * 100
    no_slang_percent = (no_slang_posts["label"] == label).sum() / len(no_slang_posts) * 100
    print(f"{label}      {round(slang_percent, 1)} %    {round(no_slang_percent, 1)} %") #sorry the print is ugly!

label      slang %    no-slang %
angry      45.1 %    29.6 %
happy      20.8 %    19.5 %
sad      16.0 %    18.0 %
fear      6.2 %    4.3 %
surprise      6.9 %    7.6 %
neutral      5.0 %    21.0 %


## 5: Summary of saved files

In [27]:
# List every file we saved, with its size
for folder in [raw_folder, processed_folder]:
    print(folder)
    # os.walk goes through a folder and all the folders inside it
    for current_dir, subfolders, file_names in os.walk(folder):
        for file_name in sorted(file_names):
            full_path = os.path.join(current_dir, file_name)
            size_in_kb = os.path.getsize(full_path) / 1024
            short_path = os.path.relpath(full_path, folder)
            print("   ", short_path, "-", round(size_in_kb), "KB")

/Users/cierrajagan/github/W266-slang-emotion-mt/data/raw
    hadqaet/annotated_dataset.xlsx - 1319 KB
    hadqaet/post-edits70p.xlsx - 955 KB
    smp2020_ewect/usual_eval_labeled.txt - 332 KB
    smp2020_ewect/usual_test_labeled.txt - 837 KB
    smp2020_ewect/usual_train.txt - 4678 KB
    smp2020_ewect/virus_eval_labeled.txt - 375 KB
    smp2020_ewect/virus_test_labeled.txt - 539 KB
    smp2020_ewect/virus_train.txt - 1580 KB
    cheer/data_cheating_filtered.py - 10115 KB
/Users/cierrajagan/github/W266-slang-emotion-mt/data/processed
    cheer_examples.json - 8305 KB
    cheer_lexicon.csv - 1036 KB
    hadqaet_annotated.csv - 1753 KB
    hadqaet_keywords.csv - 1281 KB
    hadqaet_post_edits.csv - 1733 KB
    smp2020_ewect.csv - 8082 KB
